# Model Fine-tuning Guide using LLM-Training

This notebook provides a step-by-step guide to fine-tune large language models using the `llm-training` repository.

## About this Tutorial

This fine-tuning workflow is powered making use of **[LitGPT](https://github.com/Lightning-AI/litgpt)**, a high-performance library by Lightning AI for pretraining, fine-tuning, and deploying 20+ large language models.

**Why LitGPT?**
- High-performance implementations optimized for speed and efficiency
- Memory-efficient techniques (LoRA, QLoRA, quantization)
- Powered by Lightning Fabric for distributed training
- Support a lot of popular LLM architectures
- Production-ready deployment tools

**Additional Resources:**
- **Official LitGPT Repository**: [https://github.com/Lightning-AI/litgpt](https://github.com/Lightning-AI/litgpt)
- **LoRA Fine-tuning Tutorial**: [https://github.com/Lightning-AI/litgpt/blob/main/tutorials/finetune_lora.md](https://github.com/Lightning-AI/litgpt/blob/main/tutorials/finetune_lora.md)
- **More Tutorials**: Check the `tutorials/` folder in the LitGPT repository for guides on:
  - Different fine-tuning methods (LoRA, QLoRA, Adapters, Full fine-tuning)
  - Pretraining from scratch
  - Custom dataset preparation
  - Quantization techniques

---

## Prerequisites

### Dataset Format

**Important:** Your dataset must be in **Alpaca format**. The Alpaca format is a standard JSON structure for instruction-following datasets:

```json
[
  {
    "instruction": "What is the capital of France?",
    "input": "",
    "output": "The capital of France is Paris."
  },
  {
    "instruction": "Translate the following to Spanish:",
    "input": "Hello, how are you?",
    "output": "Hola, ¿cómo estás?"
  }
]
```

### Dataset Sources

You can obtain your dataset in two ways:

1. **From HuggingFace Hub**: Download a pre-existing Alpaca-formatted dataset from HuggingFace
   - Example: `esa-sceva/satcom-synth-qa`, `esa-sceva/satcom-synth-qa-cot`, `yahma/alpaca-cleaned`, etc.
   - We'll show you how to download and convert it in Step 0

2. **Local Dataset**: Upload your own Alpaca-formatted JSON file
   - Ensure it follows the structure shown above
   - Save it as `litgpt_data.json` or update the path in the `.env` file accordingly

---

### Notebook vs Terminal Commands

**Important Note:** This notebook contains commands that work in Jupyter notebooks. Some traditional Linux commands (like `nano`, `vim`, etc.) are interactive text editors that **don't work in notebook cells**.

**Key Differences:**
- Use `%cd directory` instead of `!cd directory` - the `%cd` command persists across cells
- Use `%%writefile` magic command or Python to create files

For creating files, we'll use:
- `%%writefile` command (Jupyter's way to create files)
- Python code to write files

---

Let's begin the fine-tuning process!


## Step 0: Download and Prepare Your Dataset

Before we start fine-tuning, we need a dataset. This step shows you how to download a dataset from HuggingFace and prepare it in Alpaca format.

### Option 1: Download from HuggingFace

We'll use the `esa-sceva/satcom-synth-qa` dataset as an example.


In [ ]:
# Install required packages for dataset handling
%pip install datasets huggingface_hub -q

In [ ]:
from datasets import load_dataset
import json

# Download the dataset from HuggingFace
print("Downloading dataset from HuggingFace...")

# ===== FOR PUBLIC DATASETS (no authentication needed) =====
# dataset = load_dataset("esa-sceva/satcom-synth-qa")

# ===== FOR PRIVATE/GATED DATASETS (authentication required) =====
# Option 1: Pass token directly (RECOMMENDED - replace with your token)
dataset = load_dataset("esa-sceva/satcom-synth-qa", token="[HF_TOKEN]")

# Option 2: Login first, then load
# from huggingface_hub import login
# login(token="[HF_TOKEN]")  # Login once
# dataset = load_dataset("esa-sceva/satcom-synth-qa")  # No token parameter needed

print(f"Dataset loaded successfully!")
print(f"  Train samples: {len(dataset['train'])}")
print(f"\nDataset structure:")
print(dataset)
print(f"\nExample from dataset:")
print(dataset['train'][0])


Convert now the dataset in Alpaca format, as explained before.

In [ ]:
# Convert to row-wise records
train_data = dataset['train'].to_dict()
train_records = [
    {k: v[i] for k, v in train_data.items()} 
    for i in range(len(dataset['train']))
]

# Transform to Alpaca-style
alpaca_records = [
    {
        "instruction": record["question"],
        "input": "",
        "output": record["answer"]
    }
    for record in train_records
]

# Save as JSON
json_path = "dataset.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(alpaca_records, f, ensure_ascii=False, indent=2)

print(f"Alpaca-style dataset saved at: {json_path}")
print(f"Total samples: {len(alpaca_records)}")


## Step 1: Clone the Repository

First, we need to clone the `llm-training` repository from the [esa-sceva](https://github.com/esa-sceva) organization on GitHub. This repository contains all the necessary tools and scripts for fine-tuning.


In [ ]:
!git clone https://[GITHUB_TOKEN]@github.com/esa-satcomllm/llm-training.git

## Step 2: Navigate to the Repository

Change directory to the cloned repository using `%cd` and move the created dataset there.

**Important:** Unlike `!cd`, the `%cd` command changes the working directory for all subsequent cells!


In [ ]:
!mv dataset.json llm-training/dataset.json

In [ ]:
%cd llm-training

## Step 3: Explore Available Models with LitGPT

Before configuring your training, you can explore which models are available for download using `litgpt`.

### List All Available Models

The `litgpt download list` command shows you all models supported by LitGPT from various providers.

In [ ]:
%pip install litgpt -q

In [ ]:
!litgpt download list

### Understanding `litgpt download list`

This command shows you:
- **All models** supported by LitGPT
- Models from various providers: Meta (Llama), Mistral, Google (Gemma), Microsoft (Phi), and more
- Different model sizes (from 1B to 70B+ parameters)

**Popular models you might see:**
- `meta-llama/Meta-Llama-3.1-8B-Instruct`
- `mistralai/Mistral-7B-Instruct-v0.2`
- `mistralai/Mistral-Small-24B-Instruct-2501`
- `microsoft/phi-2`
- `google/gemma-2b`

**Choose a model based on:**
1. **Available GPU memory** (smaller models for limited resources)
2. **Task requirements** (instruction-tuned models work better for QA tasks)
3. **License restrictions** (some models require approval from HuggingFace)


## Step 4: Configure Environment Variables

Create a `.env` file to store your configuration. This file contains important parameters for the fine-tuning process:

- **JSON_PATH**: Path to your Alpaca-formatted dataset
- **MODEL_NAME**: The HuggingFace model you want to fine-tune (choose from `litgpt download list`)
- **HF_TOKEN**: Your HuggingFace access token (get it from https://huggingface.co/settings/tokens)
- **CONFIG_PATH**: Path to the configuration file (`config/lora.yaml` or `config/finetune.yaml`)
- **WANDB_TOKEN**: Your Weights & Biases token for experiment tracking
- **WANDB_NAME**: Name for your experiment in W&B
- **VAL_SPLIT**: Validation split ratio (0.05 = 5% of data for validation)
- **OUT_DIR**: Output directory for the fine-tuned model

**Update the values below** with your own tokens and paths:


In [ ]:
%%writefile ./.env
JSON_PATH=./dataset.json
MODEL_NAME=meta-llama/Meta-Llama-3.1-8B-Instruct
HF_TOKEN=[HF_TOKEN]
CONFIG_PATH=config/lora.yaml
WANDB_TOKEN=[WANDB_TOKEN]
WANDB_NAME=llama-8B
VAL_SPLIT=0.05
OUT_DIR=out/finetune/meta-llama/Meta-Llama-3.1-8B-Instruct

**Note:** The `%%writefile` magic command will create the `.env` file in the `llm-training` directory with the configuration above.

### Alternative Method (Using Python)

If `%%writefile` doesn't work in your environment, you can also create the file using Python:



In [ ]:
# Alternative: Create .env file using Python
env_content = """JSON_PATH=./dataset.json
MODEL_NAME=meta-llama/Meta-Llama-3.1-8B-Instruct
HF_TOKEN=[HF_TOKEN]
CONFIG_PATH=config/lora.yaml
WANDB_TOKEN=[WANDB_TOKEN]
WANDB_NAME=llama-8B
VAL_SPLIT=0.05
OUT_DIR=out/finetune/meta-llama/Meta-Llama-3.1-8B-Instruct
"""

with open('llm-training/.env', 'w') as f:
    f.write(env_content)

print(".env file created successfully!")

### If Running in a Terminal (Not a Notebook)

If you're running these commands in a terminal instead of a notebook, you can manually edit the `.env` file:

```bash
nano llm-training/.env
```

Then paste the configuration and save with `Ctrl+O`, `Enter`, `Ctrl+X`.


## Step 5: Install Dependencies

Run the installation script which sets up a virtual environment and installs all required Python packages.


In [ ]:
!pip install --quiet -r requirements.txt

## Step 6: Download the Model

Download the pre-trained model from HuggingFace. This will download the model specified in your `.env` file (MODEL_NAME).

In [ ]:
!make model

## Step 7: Understanding Configuration Files (lora.yaml vs finetune.yaml)

The training behavior is controlled by YAML configuration files in the `config/` directory. These configurations are based on **[LitGPT's fine-tuning system](https://github.com/Lightning-AI/litgpt/blob/main/tutorials/finetune_lora.md)**.

> **For more details**, see the official [LitGPT LoRA Fine-tuning Tutorial](https://github.com/Lightning-AI/litgpt/blob/main/tutorials/finetune_lora.md)

There are two main configuration options:

### 1. `lora.yaml` - LoRA Fine-tuning (Recommended)

**What is LoRA?**
- **Lo**w-**R**ank **A**daptation
- Efficient fine-tuning method that only trains a small number of additional parameters
- **Advantages:**
  - Requires much less GPU memory
  - Faster training
  - Smaller checkpoint files (only saves adapter weights, not the full model)
  - Often achieves similar performance to full fine-tuning
- **Use when:** You have limited GPU memory or want faster iterations

### 2. `finetune.yaml` - Full Fine-tuning

- Trains **all** model parameters
- **Advantages:**
  - Potentially better performance on specific tasks
  - More flexibility in how the model adapts
- **Disadvantages:**
  - Requires much more GPU memory
  - Slower training
  - Large checkpoint files (saves entire model)
- **Use when:** You have sufficient GPU resources and need maximum adaptation

### Configuration Comparison

| Aspect | LoRA (`lora.yaml`) | Full Fine-tuning (`finetune.yaml`) |
|--------|-------------------|-----------------------------------|
| **GPU Memory** | Low (2-8GB) | High (24GB+) |
| **Training Speed** | Fast | Slow |
| **Checkpoint Size** | Small (~100MB) | Large (GBs) |
| **Performance** | Good | Potentially better |
| **Best for** | Most use cases | Specialized tasks |


### Key Parameters in `lora.yaml`

The `lora.yaml` file contains many configurable parameters. Let's break down the most important ones:

#### Training Parameters (`train` section):
```yaml
train:
  epochs: 2                    # Number of times to iterate through the entire dataset
                               # ↑ Increase for better convergence, decrease if overfitting
  
  global_batch_size: 128       # Total batch size across all GPUs
                               # ↑ Larger = more stable but slower
  
  micro_batch_size: 4          # Batch size per GPU
                               # ↓ Decrease if you run out of GPU memory
  
  lr_warmup_steps: 50          # Learning rate warmup steps
  
  max_seq_length: 512          # Maximum sequence length
                               # ↑ Increase for longer documents, ↓ decrease to save memory
  
  save_interval: 200           # Save checkpoint every N steps
  
  log_interval: 1              # Log metrics every N steps
```

#### LoRA-specific Parameters:
```yaml
lora_r: 32                     # LoRA rank (dimensionality of adapter)
                               # ↑ Higher = more capacity but slower, typical: 8-64

lora_alpha: 16                 # LoRA scaling factor
                               # Usually set to lora_r or 2*lora_r

lora_dropout: 0.05             # Dropout rate for LoRA layers

# Which parts of the model to apply LoRA to:
lora_query: true               # Apply to query weights (recommended)
lora_key: false                # Apply to key weights
lora_value: true               # Apply to value weights (recommended)
lora_projection: false         # Apply to projection layer
lora_mlp: false                # Apply to MLP layers
lora_head: false               # Apply to output head
```

#### Data Parameters:
```yaml
data:
  json_path: /path/to/data.json    # Path to your Alpaca-format dataset
  mask_prompt: false                # Whether to mask the prompt in loss calculation
  prompt_style: alpaca              # Prompt template style
  val_split_fraction: 0.05          # Fraction of data for validation (5%)
  num_workers: 4                    # Number of data loading workers
```

#### Optimizer Parameters:
```yaml
optimizer:
  class_path: torch.optim.AdamW
  init_args:
    lr: 0.0001                 # Learning rate
                               # ↑ Larger = faster learning but less stable
    weight_decay: 0.0          # L2 regularization
    betas: [0.9, 0.95]         # Adam beta parameters
```

#### Hardware Parameters:
```yaml
devices: 1                     # Number of GPUs to use
num_nodes: 1                   # Number of compute nodes
precision: bf16-true           # Training precision (bf16-true, bf16-mixed, or 32-true)
                               # bf16 = faster and saves memory
quantize: null                 # Optional quantization (nf4, fp4, int8-training)
```

#### Evaluation Parameters:
```yaml
eval:
  interval: 100                # Evaluate every N optimizer steps
  max_new_tokens: 100          # Tokens to generate during evaluation
  max_iters: 100               # Number of eval iterations
  initial_validation: false    # Evaluate before training
  final_validation: true       # Evaluate after training
```

#### Logging:
```yaml
logger_name: wandb             # Logger: wandb, tensorboard, or csv
seed: 1337                     # Random seed for reproducibility
```


### Quick Reference: When to Adjust Parameters

| Problem | Solution |
|---------|----------|
| **Out of GPU memory** | ↓ Decrease `micro_batch_size`, `max_seq_length`, or `lora_r` |
| **Underfitting** | ↑ Increase `epochs`, `lora_r`, or `lr` |
| **Overfitting** | ↓ Decrease `epochs`, increase `lora_dropout` or `weight_decay` |
| **Training too slow** | ↑ Increase `micro_batch_size` or decrease `save_interval` |
| **Unstable training** | ↓ Decrease `lr` or increase `lr_warmup_steps` |
| **Need longer context** | ↑ Increase `max_seq_length` (but uses more memory) |

### How to Modify Configuration Parameters

You have two options to modify the configuration:

**Option 1:** Edit the YAML file directly

**Option 2:** Use Python to modify the config programmatically (shown below):


In [ ]:
# Example: Modify lora.yaml parameters
%pip install pyyaml -q

import yaml

# Read the existing config
with open('config/lora.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Current configuration:")
print(f"  Epochs: {config['train']['epochs']}")
print(f"  Micro batch size: {config['train']['micro_batch_size']}")
print(f"  Max sequence length: {config['train']['max_seq_length']}")
print(f"  LoRA rank: {config['lora_r']}")
print(f"  Learning rate: {config['optimizer']['init_args']['lr']}")

# Modify parameters as needed
config['train']['epochs'] = 3  # Train for 3 epochs instead of 2
config['train']['micro_batch_size'] = 2  # Reduce if OOM
config['train']['max_seq_length'] = 1024  # Increase for longer texts
config['lora_r'] = 64  # Increase LoRA rank for more capacity
config['optimizer']['init_args']['lr'] = 0.0002  # Adjust learning rate

# Save modified config
with open('config/lora_custom.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("\n✓ Custom configuration saved to config/lora_custom.yaml")
print("\nModified parameters:")
print(f"  Epochs: {config['train']['epochs']}")
print(f"  Micro batch size: {config['train']['micro_batch_size']}")
print(f"  Max sequence length: {config['train']['max_seq_length']}")
print(f"  LoRA rank: {config['lora_r']}")
print(f"  Learning rate: {config['optimizer']['init_args']['lr']}")
print("\nUpdate CONFIG_PATH in .env to 'config/lora_custom.yaml' to use these settings")


## Step 8: Start Training

Now you're ready to start the fine-tuning process! This command will:
- Load your dataset from the path specified in `.env`
- Apply LoRA (Low-Rank Adaptation) for efficient fine-tuning
- Train the model using the configuration in `config/lora.yaml`
- Save checkpoints to the output directory specified in `.env`
- Log metrics to Weights & Biases if configured

**Note:** Training time will vary depending on:
- Dataset size
- Model size
- Hardware (GPU type and memory)
- Training configuration (epochs, batch size, etc.)


In [ ]:
!pwd

In [ ]:
!make train

## After Training

### Accessing Your Fine-tuned Model

Once training is complete, your fine-tuned model will be saved in the output directory specified in your `.env` file (OUT_DIR). For this example, it will be:

```
out/finetune/mistralai/Mistral-Small-24B-Instruct-2501
```

### Monitoring Training Progress

If you configured Weights & Biases (WANDB), you can monitor:
- Training loss
- Validation loss
- Learning rate schedules
- GPU utilization
- And more...

Visit your W&B dashboard at: https://wandb.ai/

### Tips for Successful Fine-tuning

1. **Dataset Quality**: Ensure your Alpaca-formatted dataset is high-quality and diverse
2. **Validation Split**: Use an appropriate validation split (typically 5-10%) to monitor overfitting
3. **Monitor Metrics**: Keep an eye on both training and validation loss
4. **Experiment**: Try different hyperparameters in `config/lora.yaml` if needed
5. **GPU Memory**: If you encounter OOM errors, consider:
   - Reducing batch size
   - Using gradient accumulation
   - Adjusting LoRA rank

### Next Steps

After successful fine-tuning, you can:
- Test your model on new prompts using `litgpt generate` or `litgpt chat`
- Push the model to HuggingFace Hub
- Deploy the model for inference using `litgpt serve`
- Further fine-tune on additional datasets
- Merge LoRA weights with base model (see [LitGPT docs](https://github.com/Lightning-AI/litgpt/blob/main/tutorials/finetune_lora.md#merging-lora-weights-optional))

### Learn More

**Explore more with LitGPT:**
- [Official LitGPT Repository](https://github.com/Lightning-AI/litgpt) - Main documentation and examples
- [LoRA Fine-tuning Tutorial](https://github.com/Lightning-AI/litgpt/blob/main/tutorials/finetune_lora.md) - Detailed guide on LoRA/QLoRA
- [Quantization Guide](https://github.com/Lightning-AI/litgpt/blob/main/tutorials/quantize.md) - Reduce memory usage
- [Deployment Tutorial](https://github.com/Lightning-AI/litgpt/blob/main/tutorials) - Deploy your models
- [Evaluation Guide](https://github.com/Lightning-AI/litgpt/blob/main/tutorials) - Benchmark your models
- [All LitGPT Tutorials](https://github.com/Lightning-AI/litgpt/tree/main/tutorials) - Complete learning resources

---

Happy fine-tuning! 🚀

*Powered using [LitGPT](https://github.com/Lightning-AI/litgpt)*
